## **Project Overview: Smart Web Scrapper**

**Smart Web Scrapper** is designed to efficiently extract data from any given URL and generate insightful summaries. The project aims to streamline web data collection and transform raw content into meaningful, actionable insights.

In [2]:
#environment check
import sys
import platform

print(f"Python version: {sys.version}")
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Executable: {sys.executable}")

Python version: 3.14.5 (main, Jun  2 2026, 22:34:49) [MSC v.1944 64 bit (AMD64)]
Platform: Windows 11
Executable: c:\AI Projects\ai-engineering\webscraper\.venv\Scripts\python.exe


In [3]:
import httpx

try:
    r = httpx.get('https://webscraper.io/test-sites/product/mercedes-benz-w123-280e-1955-c001')
    print(r.status_code)

except Exception as e:
    print(f"An error occurred: {e.type()}")

200


In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r,'html.parser')
print(soup.body.prettify())

<body class="ws-v2">
 <!-- Google Tag Manager (noscript) -->
 <noscript>
  <iframe height="0" src="https://www.googletagmanager.com/ns.html?id=GTM-NVFPDWB" style="display:none;visibility:hidden" width="0">
  </iframe>
 </noscript>
 <!-- End Google Tag Manager (noscript) -->
 <script nonce="MfaobzmZiQUAAuRpLnHbVy0FdG3diZZn" type="text/javascript">
  !function (e, t, n) {
			function a() {
				var e = t.getElementsByTagName("script")[0],
					n = t.createElement("script");
				n.type = "text/javascript", n.async = !0, n.src = "https://beacon-v2.helpscout.net", e.parentNode.insertBefore(n, e)
			}

			if (e.Beacon = n = function (t, n, a) {
				e.Beacon.readyQueue.push({method: t, options: n, data: a})
			}, n.readyQueue = [], "complete" === t.readyState) return a();
			e.attachEvent ? e.attachEvent("onload", a) : e.addEventListener("load", a, !1)
		}(window, document, window.Beacon || function () {
		});
 </script>
 <script nonce="MfaobzmZiQUAAuRpLnHbVy0FdG3diZZn" type="text/javascript">

| Field       | Tag  | Class (or id)       | Example Value                                                                                           |
|-------------|------|---------------------|--------------------------------------------------------------------------------------------------------|
| title       | h2   | title               | Mercedes-Benz W123 280E 1955                                                                           |
| price       | h3   | amount              | 228 531                                                                                                |
| description | p    | description mt-5    | The W123 280E, conceived without compromise and manufactured to tolerances that proved themselves over decades of use. Presented here in project condition. |

In [23]:
def extract(soup, tag, class_name, attr=None):

    element = soup.find(tag, class_ = class_name)

    if element is None:
        return None
    
    if attr is not None:
        return element.get(attr)
    
    return element.text.strip()

title = extract(soup, 'h2', 'title')
price = extract(soup, 'span', 'amount', attr='data-price')
description = extract(soup, 'p', 'description')

if price is not None:
    price = int(price)

print("Title:", title)
print("Price:", price)
print("Description:", description)

Title: Mercedes-Benz W123 280E 1955
Price: 228511
Description: The W123 280E, conceived without compromise and manufactured to tolerances that proved themselves over decades of use. Presented here in project condition.


In [5]:
rs = httpx.get('https://www.therundown.ai/news/mit-living-muscle-swimming-robot')
print(rs.status_code)


200


In [25]:
soup = BeautifulSoup(rs.text, 'html.parser')
before = len(soup.get_text())

# Add common unwanted tags to the list
unwanted_tags = [
    "script", "style", "footer", "nav", "aside", "noscript", "input", "button", "svg", "canvas",
    "dialog", "iframe"
]

for tag in soup.find_all(unwanted_tags):
    tag.decompose()

print()

after = len(soup.get_text())

print("before" , before)
print("after" , after)

print(f"Removed {before - after} characters")

print(soup.get_text(separator="\n", strip=True))





before 4960
after 3946
Removed 1014 characters
MIT builds a thin swimming robot powered by living muscle | The Rundown AI
The Rundown AI homepage
University Platform →
Robotics
/
News & analysis
MIT builds a thin swimming robot powered by living muscle
MIT’s thin robot swims with living muscle cells and steers with light. It could lead to smaller, cheaper robots, though it still tires in the lab.
By
The Rundown Editorial Team
Reviewed by
Kelly Pitts
·
October 2, 2026
·
3
min read
Image source: Melanie Gonick, MIT
MIT engineers have built a robot the size of a stick of gum that swims with living mouse muscle cells. In a demonstration
announced on September 29
, researchers steered it through a maze in a Petri dish by aiming blue light at its fins.
The swimmer,
featured in The Rundown’s October 1 edition
, combines thin muscle films with a soft gel support. The team reports that these films produce more force for their volume than earlier muscle designs, which could help make future rob

In [26]:
article_text = soup.get_text(separator = "\n", strip=True)
search_phrase = "paper-thin robot swims through watery maze"

if search_phrase in article_text:
    print("Found the phrase in the article.")
else:
    print("Phrase not found in the article.")


Found the phrase in the article.


In [6]:
# Here's what I found when inspecting the HTML:
#
# Content tags to keep: h2, h3, h4, p, li
#   - These tags usually hold the primary content or structured information that we want to extract.
#
# Junk tags to drop: footer, nav, figure, aside, form, script, style
#   - These tags are mostly used for navigation, layout, formatting, images, or scripts – typically not part of the main readable page content.
#
# Is this right?
# - This method effectively captures most readable content. Retaining <li> tags, especially within <ul>/<ol>, is a good way to ensure list data is not missed.
# - You might want to exclude additional tags like <header> or <section> if you notice irrelevant content from those.
# - "figure" may sometimes contain images and captions. If you want only text, removing it is good; if image captions are important, more selective extraction may help.
#
# Known tradeoffs / rtradeoffs:
# - Some valuable content inside dropped tags (e.g., important captions in "figure", critical info tucked in <aside>) may be lost.
# - Retaining only certain tags improves focus but might omit context or related media information.
# - Dropping too many tags could unintentionally strip introductory, summary, or related links sections, so always verify with actual page samples.



HEADERS = {
    "User-Agent": "MySmartScraper/0.1 (aibhidev@gmail.com)",
}

def fetch_and_clean(url):

    try:
        response = httpx.get(url, headers=HEADERS)
    except httpx.RequestError as e:
        print(f"An error occurred: {type(e).__name__} while fetching {url}")
        return None

    if response.status_code != 200:
            print(f"Error: Failed to retrieve the web page. Status code: {response.status_code}. URL: {url}")
            return None

    soup = BeautifulSoup(response.text, "html.parser")

    BLOCKLIST_TAGS = [
        "script", "style", "footer", "nav", "aside", "noscript", "input", "button", "svg", "canvas",
        "dialog", "iframe", "figure", "header", "link", "table"
         ]
    
    ALLOWLIST = [
        "h1", "h2", "h3", "h4", "p", "li"
    ]

    for heading in soup.find_all(["h2", "h3", "h4"]):
        if heading.get_text(strip=True) in ["References", "Further reading", "See also", "External links"]:
            block = heading.find_parent("div", class_="mw-heading") or heading
            for sibling in list(block.find_next_siblings()):
                sibling.decompose()
            block.decompose()
            break



    for tag in soup.find_all(BLOCKLIST_TAGS):
        tag.decompose()

    extracted_texts = []

    for tag in soup.find_all(ALLOWLIST):
        text = tag.get_text(separator=" ", strip=True)
        if text:
            extracted_texts.append(text)

    return "\n".join(extracted_texts).strip()

    

cleaned_text = fetch_and_clean('https://en.wikipedia.org/wiki/Kangaroo')

print(cleaned_text)



Kangaroos are marsupials from the subfamily Macropodinae (macropods, meaning "large foot"). In common use, the term is used to describe the largest species from this family, the red kangaroo , as well as the antilopine kangaroo , eastern grey kangaroo , and western grey kangaroo . [ 1 ] Kangaroos are indigenous to Australia and New Guinea . The Australian government estimated that 42.8 million kangaroos lived within the commercial harvest areas of Australia in 2019, down from 53.2 million in 2013. [ 2 ]
As with the terms " wallaroo " and " wallaby ", "kangaroo" refers to a paraphyletic grouping of species. All three terms refer to members of the same taxonomic family, Macropodidae, and are distinguished according to size. The largest species in the family are called "kangaroos" and the smallest are generally called "wallabies". The term "wallaroos" refers to species of an intermediate size. [ 3 ] There are also the tree-kangaroos , another type of macropod which inhabit the upper branc

In [84]:
HEADERS = {
    "User-Agent": "MySmartScraper/0.1 (aibhidev@gmail.com)",
}

response01 = httpx.get('https://www.therundown.ai/news/mit-living-muscle-swimming-robot', headers = HEADERS)

soup = BeautifulSoup(response01.text, "html.parser")

for p_tag in soup.find_all("p"):
    print(p_tag.get_text(strip=True))

MIT’s thin robot swims with living muscle cells and steers with light. It could lead to smaller, cheaper robots, though it still tires in the lab.
MIT engineers have built a robot the size of a stick of gum that swims with living mouse muscle cells. In a demonstrationannounced on September 29, researchers steered it through a maze in a Petri dish by aiming blue light at its fins.
The swimmer,featured in The Rundown’s October 1 edition, combines thin muscle films with a soft gel support. The team reports that these films produce more force for their volume than earlier muscle designs, which could help make future robots smaller and cheaper to build.
Each of the robot’s two gel fins carries mouse muscle cells genetically engineered to contract when exposed to blue light. Researchers control the fins independently, adjusting light pulses to change direction and speed. In the maze trial, they moved the light source by hand. The swimmer reachedroughly four body lengths per minute.
Microscop